# Training a Tiny LLM From Scratch on 1-Digit Addition — Reference Notebook

> **Reference notebook.** See [`reasoning.md`](./reasoning.md) for why this toy arithmetic task is used
> as an example, and [`tiny_addition_llm_gradio_deploy.ipynb`](./tiny_addition_llm_gradio_deploy.ipynb)
> for loading and serving the model this notebook saves.

**Methods covered:**
- Building a **tiny GPT-2-architecture model from scratch** (`AutoConfig` + `AutoModelForCausalLM.from_config`)
  — random initialization, not a fine-tune of pretrained GPT-2 weights
- A hand-rolled, fixed-vocabulary tokenizer (`AdditionTokenizer`) for a task far narrower than natural language
- Framing a supervised next-token-prediction dataset for arithmetic (`AdditionDataset`)
- A manual training loop with `accelerate.Accelerator` and masked cross-entropy loss
- A hand-written autoregressive generation loop (no `.generate()`) and a simple accuracy eval
- Saving the trained model with `save_pretrained`

**Use this as a reference when:** you want to see how a transformer is built and trained **from scratch**
on a minimal synthetic task — useful for understanding what training actually does under the hood, or for
prototyping a tiny custom-vocabulary model.

**Don't use this as a reference for:** fine-tuning a pretrained model (see
[`06-07-08-transfer-learning-fine-tuning`](../06-07-08-transfer-learning-fine-tuning/fine_tuning.md) or
[`09-legal-assistant-llm-finetuning`](../09-legal-assistant-llm-finetuning/README.md) instead), or
production-grade tokenization (a real tokenizer handles arbitrary text, not a 13-token closed vocabulary).


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from accelerate import Accelerator
from torch.utils.data import Dataset
from transformers import AutoConfig, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# vocab_size=13 -- the model only ever needs to see: '+', '=', a pad token, and digits 0-9.
# n_ctx=6 -- context window sized for exactly "d + d = dd" (4 input tokens + 2 output digits), nothing more.
# n_head/n_layer are cut down from GPT-2's defaults (12/12) since the task has none of the complexity
# that justifies a large model.
vocab_size = 13
sequence_length = 4
result_length = 2
context_length = sequence_length + result_length

config = AutoConfig.from_pretrained(
    "gpt2", vocab_size=vocab_size, n_ctx=context_length, n_head=4, n_layer=2,
)

# from_config (not from_pretrained) builds the model with GPT-2's *architecture* but random weights --
# nothing is fine-tuned here. This model has never seen language and starts knowing nothing.
model = AutoModelForCausalLM.from_config(config)

num_params = sum(t.numel() for t in model.parameters())
print(f"Model size: {num_params / 1000**2:.1f}M parameters")


In [ ]:
class AdditionTokenizer:
    """Fixed 13-token vocabulary: '+', '=', a pad token, and the digits 0-9.

    Unlike a real tokenizer (subword/BPE, open vocabulary), this only ever splits on whitespace and
    looks each whitespace-separated piece up in a fixed dict -- every input token must already be
    exactly one of these 13 symbols, single-digit numbers included.
    """

    def __init__(self):
        vocab = ["+", "=", "-1", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
        self.pad_token = "-1"
        self.encoder = {str(v): i for i, v in enumerate(vocab)}
        self.decoder = {i: str(v) for i, v in enumerate(vocab)}
        self.pad_token_id = self.encoder[self.pad_token]

    def decode(self, token_ids):
        return " ".join(self.decoder[t] for t in token_ids)

    def __call__(self, text):
        # No subword fallback: a piece that isn't in the vocab (e.g. the multi-digit "14") raises KeyError.
        return [self.encoder[t] for t in text.split()]


tokenizer = AdditionTokenizer()

print(tokenizer("1 + 1 = 2"))          # every token is a single recognized symbol -- works
print(tokenizer("9 + 5 = 1 4"))        # "14" must be spelled as two separate digit tokens -- works
try:
    tokenizer("9 + 5 = 14")            # "14" as one piece is not a vocab entry -- fails
except KeyError as e:
    print(f"KeyError: {e} -- multi-digit results must be space-separated digits, not one token")


In [ ]:
class AdditionDataset(Dataset):
    """Generates (input, target) pairs on the fly: input '2 + 3 = 0' -> target '+ 3 = 0 5'.

    The dataset never overlaps in practice (it draws random operands each call), so "size" is just a
    number of steps per epoch, not a fixed number of stored examples.
    """

    def __init__(self, split, length=6):
        assert split in {"train", "test"}
        self.split = split
        self.length = length

    def __len__(self):
        return 1_000_000

    def __getitem__(self, idx):
        available_numbers = [
            int(n) for n in tokenizer.decoder.values()
            if n != tokenizer.pad_token and str(n).isnumeric()
        ]
        inp = torch.tensor(np.random.choice(available_numbers, size=result_length))
        sol = torch.tensor([int(i) for i in str(inp.sum().item())])
        # Left-pads single-digit sums (e.g. "8") to two digits ("0 8") so every target has fixed length.
        sol = torch.nn.functional.pad(sol, (1 if sol.size()[0] == 1 else 0, 0), "constant", 0)

        cat = torch.cat((inp, sol), dim=0)
        x = cat[:-1].clone()
        y = cat[1:].clone()
        # The model predicts each *next* token from the previous ones; the very first target position
        # has no preceding operand to predict from, so it's masked with the pad id and excluded from loss.
        y[:1] = int(tokenizer.pad_token)

        x = f"{x[0].item()} + {x[1].item()} = {x[2].item()}"
        y = f"-1 {y[0].item()} -1 {y[1].item()} {y[2].item()}"

        return torch.tensor(tokenizer(x)), torch.tensor(tokenizer(y))


train_dataset = AdditionDataset("train", length=sequence_length)
test_dataset = AdditionDataset("test", length=sequence_length)

x, y = train_dataset[0]
print("input: ", tokenizer.decode(x.numpy()))
print("target:", tokenizer.decode(y.numpy()))


In [ ]:
num_epochs = 2
batch_size = 100

optimizer = torch.optim.Adam(model.parameters())
dataloader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=batch_size)

# Accelerator abstracts away device placement (CPU/GPU/multi-GPU) -- .prepare() moves the model,
# optimizer, and dataloader onto whatever device is available without any device-specific code here.
accelerator = Accelerator()
model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)
model.train()


In [ ]:
print("Training the tiny LLM...")

for epoch in range(num_epochs):
    for source, targets in dataloader:
        optimizer.zero_grad()

        # ignore_index=pad_token_id -- the masked first target position (see AdditionDataset above)
        # must not contribute to the loss, since there's nothing meaningful the model could predict there.
        loss = F.cross_entropy(
            model(source).logits.flatten(end_dim=1),
            targets.flatten(end_dim=1),
            ignore_index=tokenizer.pad_token_id,
        )
        accelerator.backward(loss)
        optimizer.step()

    print(f"Epoch {epoch + 1}/{num_epochs} -- loss: {loss.item()}")

print("Training complete.")


In [ ]:
def predict(text, solution_length=result_length, model=model):
    # This tiny custom-vocab model isn't a good fit for the standard `.generate()` utilities (built
    # around real tokenizers and text vocabularies), so decoding is done by hand: predict one token,
    # append it to the input, repeat.
    model.eval()
    input_ids = torch.tensor(tokenizer(text)).to(accelerator.device).unsqueeze(0)

    solution = []
    for _ in range(solution_length):
        logits = model(input_ids).logits[0, -1]
        predicted = logits[:vocab_size].argmax()  # restrict to the real vocab (drop any GPT-2-config padding logits)
        input_ids = torch.cat((input_ids, predicted.unsqueeze(0).unsqueeze(0)), dim=1)
        solution.append(predicted.cpu().item())

    return tokenizer.decode(solution)


def evaluate_accuracy(num_samples=1000):
    correct = 0
    for i in range(num_samples):
        input_ids, target_ids = test_dataset[i]
        input_text = tokenizer.decode(input_ids.cpu().numpy()[:sequence_length])
        target_text = tokenizer.decode(target_ids.cpu().numpy()[sequence_length - 1:])
        if predict(input_text) == target_text:
            correct += 1
    print(f"Accuracy: {correct / num_samples}")


evaluate_accuracy(num_samples=1000)


In [ ]:
# Saved here so tiny_addition_llm_gradio_deploy.ipynb can reload it with
# AutoModelForCausalLM.from_pretrained("./tiny_addition_llm/") -- run this notebook first.
model.save_pretrained("./tiny_addition_llm/")
